# Per-token probe activation viewer (Blog 4)

Renders per-token classifier scores for the three Apollo published probes
(`roleplaying`, `followup`, `instructed_pairs` / RePE) over a handful of
representative AF transcripts. Output formats:

1. **Interactive HTML** (CircuitsVis `colored_tokens_multi`) for a standalone
   viewer hosted via GitHub Pages, linked from the LessWrong post.
2. **Static PNG snapshots** (matplotlib) for inline blog figures, since
   LessWrong sanitises iframes and `<script>` tags.

Both share the same input JSONL produced by
`scripts/dev/save_per_token_scores.py` on the pod (one row per transcript,
schema documented in cell 2).

## Workflow

1. **On pod**: `uv run python scripts/dev/save_per_token_scores.py --probes roleplaying followup instructed_pairs`
   produces `data/processed/per_token_scores.jsonl` (~50 MB for 24 transcripts x 3 probes).
2. **`make hf-push-data`** on pod, **`make hf-pull-data`** locally.
3. Run this notebook top-to-bottom. Outputs land in `docs/figures/probe_tokens/` (PNGs)
   and `docs/viewer/probe_tokens.html` (interactive HTML).
4. Commit `docs/viewer/` to a `gh-pages` branch (or configure GitHub Pages
   from `main /docs`) to serve the viewer.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import circuitsvis as cv
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display

# Paths (relative to repo root; adjust if running from elsewhere)
SCORES_PATH = Path("../data/processed/per_token_scores.jsonl")
PNG_OUTPUT_DIR = Path("../docs/figures/probe_tokens")
HTML_OUTPUT_PATH = Path("../docs/viewer/probe_tokens.html")

PNG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Probe ordering -- mirror Figure 5's cross-probe colours
PROBES = ("roleplaying", "followup", "instructed_pairs")
PROBE_LABELS = {
    "roleplaying": "Roleplaying (surface content)",
    "followup": "Followup (deliberation structure)",
    "instructed_pairs": "RePE / Honesty (semantic)",
}

## 2. Load per-token scores JSONL

Expected schema per row (one row per transcript):

```json
{
  "sample_id": 0,
  "input_idx": 0,
  "manual_label": "af",                  // "af" | "af_partial" | "not_af"
  "is_free_tier_user": true,
  "advbench_query": "...",
  "tokens": ["<|begin_of_text|>", " You", " are", ...],   // length N
  "scores": {                                              // each length N
    "roleplaying":      [-0.21, 0.12, ...],
    "followup":         [ 0.05, 0.08, ...],
    "instructed_pairs": [ 0.30, 0.45, ...]
  },
  "sections": {                                            // [start, end) token indices
    "system":     [0, 1234],
    "user":       [1234, 1456],
    "scratchpad": [1456, 2100],
    "response":   [2100, 2350]
  }
}
```

If a section is missing (e.g. transcript was refused before scratchpad began),
the entry is absent from `sections` rather than `[start, start]`.
Scores live in **logit space** (signed; positive = probe leans toward AF).
The viewer normalises per-section via abs-max so colours read cleanly.

In [ ]:
def load_per_token_scores(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


rows = load_per_token_scores(SCORES_PATH)
print(f"Loaded {len(rows)} transcripts")
print(f"Labels: {dict((lbl, sum(1 for r in rows if r['manual_label'] == lbl)) for lbl in ('af', 'af_partial', 'not_af'))}")
rows[0].keys() if rows else None

## 3. Select representative transcripts

Pick 4-5 transcripts for the inline blog figures + viewer. Suggested mix:
- 2 canonical AF positives (one short scratchpad, one long for variety)
- 1 af_partial that the curve flips under positive steering
- 1 free-tier refusal baseline (clean negative)

In [ ]:
# Picked from the data after a candidacy analysis -- see commit message
# for the ranking criteria. The set is calibrated to make the
# "three probes, three signatures" finding visually obvious:
#
# - sid=34 (af): Followup probe peaks at +0.82 in the scratchpad while
#   roleplaying peaks at +0.50 and RePE at +0.36 -- the most striking
#   "one probe dominates" example. Visually compelling probe divergence.
# - sid=40 (af): all three probes peak ~+0.50 in the scratchpad. The
#   "three probes, same tokens, different signatures" canonical case.
# - sid=48 (not_af, "malware refusal"): substantial scratchpad (391
#   tokens) AND response (356 tokens), so both panels render clean
#   refusal-pattern contrast against the af examples above.
REPRESENTATIVE_SAMPLE_IDS = [34, 40, 48]

by_sid = {r["sample_id"]: r for r in rows}
selected = [by_sid[sid] for sid in REPRESENTATIVE_SAMPLE_IDS if sid in by_sid]
for r in selected:
    n_tokens = len(r["tokens"])
    sections = r.get("sections", {})
    prompt = (r.get("advbench_query") or "")[:70]
    print(
        f"sample_id={r['sample_id']:>3}  label={r['manual_label']:<11}  "
        f"n_tokens={n_tokens:>5}  sections={sorted(sections.keys())}  prompt={prompt!r}"
    )

## 4. Interactive rendering (CircuitsVis)

`colored_tokens_multi` shows one row per probe over the same token stream.
Hover for exact value, click a probe label to toggle isolation.

We render one panel per (transcript, section) -- splitting on the natural
boundaries (system / scratchpad / response) keeps the visualisation
readable instead of one giant block of tokens.

In [ ]:
def slice_section(row: dict[str, Any], section: str) -> tuple[list[str], np.ndarray] | None:
    """Pull tokens + (n_tokens, n_probes) score matrix for one section.

    Returns None if the section isn't present (e.g. transcript had no
    response after a pre-scratchpad refusal).
    """
    section_range = row.get("sections", {}).get(section)
    if section_range is None:
        return None
    start, end = section_range
    if end <= start:
        return None
    tokens = row["tokens"][start:end]
    scores = np.stack(
        [np.array(row["scores"][p][start:end], dtype=np.float32) for p in PROBES],
        axis=1,
    )
    return tokens, scores


def _center_per_probe(scores: np.ndarray) -> np.ndarray:
    """Subtract per-probe median so CircuitsVis's symmetric-around-zero
    colourmap reads sensibly even when Apollo's probe intercepts
    (dropped at load time) leave the raw logits far from zero.

    Same intent as the static renderer's per-panel min-max -- both
    show within-panel *relative* activation rather than absolute
    logit value.
    """
    return scores - np.median(scores, axis=0, keepdims=True)


def render_section(row: dict[str, Any], section: str) -> Any | None:
    """Render one section's tokens + per-probe scores with CircuitsVis.

    CircuitsVis ``colored_tokens_multi`` expects a torch.Tensor of shape
    (n_tokens, n_probes); we convert from numpy at the boundary.
    """
    payload = slice_section(row, section)
    if payload is None:
        return None
    tokens, scores = payload
    centered = _center_per_probe(scores)
    values_tensor = torch.from_numpy(centered.astype(np.float32))
    return cv.tokens.colored_tokens_multi(
        tokens=tokens,
        values=values_tensor,
        labels=[PROBE_LABELS[p] for p in PROBES],
    )


# Render one full transcript to verify the pipeline before batching
if selected:
    demo = selected[0]
    print(f"Demo: sample_id={demo['sample_id']}, label={demo['manual_label']}")
    print(f"AdvBench: {demo.get('advbench_query', '')[:120]}")
    for section in ("scratchpad", "response"):
        widget = render_section(demo, section)
        if widget is not None:
            display(HTML(f"<h4>{section}</h4>"))
            display(widget)

## 5. Static PNG snapshots for inline blog figures

Matplotlib `text` with per-token `bbox`, wrapping at fixed character width.
One figure per (transcript, probe) so the comparison panel shows three
differently-coloured copies of the same token stream side-by-side. Saved
into `docs/figures/probe_tokens/sample_<sid>_<section>.png`.

We score-normalise per panel by abs-max so each probe's colour scale is
honest to its own activations rather than squashed by another probe's
stronger signal.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

# Match CircuitsVis convention: red (+) -> white (0) -> blue (-)
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "af_probe", ["#1f77b4", "#ffffff", "#d62728"], N=256
)

TOKENS_PER_ROW = 18  # rough fit for a 12in figure at 9pt monospace


def _render_static_section(
    tokens: list[str],
    scores: np.ndarray,  # 1-D, length len(tokens)
    ax: plt.Axes,
    title: str,
) -> None:
    """Lay tokens out in a wrapped grid, colour each by its score.

    Normalisation: per-panel min-max into [0, 1]. We use the panel's own
    range rather than a symmetric ``[-abs_max, +abs_max]`` because
    Apollo's followup + instructed_pairs probes have non-trivial
    intercepts that we drop when loading (intercept set to 0 in
    ``_load_apollo_published_probe``). The resulting logits sit far
    from zero -- a symmetric scale paints whole panels one colour.
    Per-panel min-max shows the *relative* activation pattern within
    each probe + section, which is the interpretable quantity for
    'which tokens does this probe fire on most?'.
    """
    lo = float(scores.min())
    hi = float(scores.max())
    span = max(hi - lo, 1e-6)
    n = len(tokens)
    n_rows = int(np.ceil(n / TOKENS_PER_ROW))

    ax.set_xlim(0, TOKENS_PER_ROW)
    ax.set_ylim(-n_rows, 1)
    ax.set_axis_off()
    ax.set_title(f"{title}  (range {lo:+.2f} .. {hi:+.2f})", loc="left", fontsize=11)

    for i, (tok, val) in enumerate(zip(tokens, scores, strict=False)):
        row = i // TOKENS_PER_ROW
        col = i % TOKENS_PER_ROW
        intensity = (val - lo) / span  # [lo, hi] -> [0, 1]
        colour = DIVERGING_CMAP(intensity)
        # Strip newlines for inline display; CircuitsVis does the same
        display_tok = tok.replace("\n", "\\n").replace("\t", "\\t")
        ax.text(
            col + 0.5,
            -row,
            display_tok,
            ha="center",
            va="center",
            fontsize=8,
            family="monospace",
            bbox=dict(facecolor=colour, edgecolor="none", boxstyle="round,pad=0.15"),
        )


def save_static_panel(row: dict[str, Any], section: str, output_path: Path) -> None:
    payload = slice_section(row, section)
    if payload is None:
        return
    tokens, scores = payload

    fig, axes = plt.subplots(
        len(PROBES), 1, figsize=(12, 2.2 * len(PROBES)), dpi=180
    )
    if len(PROBES) == 1:
        axes = [axes]

    for ax, probe, probe_idx in zip(axes, PROBES, range(len(PROBES)), strict=False):
        _render_static_section(
            tokens=tokens,
            scores=scores[:, probe_idx],
            ax=ax,
            title=PROBE_LABELS[probe],
        )

    fig.suptitle(
        f"sample {row['sample_id']} ({row['manual_label']}) -- {section}",
        fontsize=13,
    )
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {output_path}")


# Generate static panels for each (transcript, section) combo. Skips sections
# that don't exist for a given transcript.
for r in selected:
    for section in ("scratchpad", "response"):
        out_path = PNG_OUTPUT_DIR / f"sample_{r['sample_id']:03d}_{section}.png"
        save_static_panel(r, section, out_path)

## 6. Standalone HTML viewer for GitHub Pages

Bundle every transcript into a single navigable HTML document. The
CircuitsVis widgets bring their own React bundle inline, so the page is
self-contained -- no CDN, no build step. Embed in a GitHub Pages site
by committing the file under `docs/` and configuring Pages to serve from
`main /docs`.

Each transcript gets a `<details>` block so the page is scannable at a
glance and readers can drill into individual transcripts on demand.

In [ ]:
from circuitsvis.utils.render import RenderedHTML


def _widget_to_html(widget: RenderedHTML | Any) -> str:
    """CircuitsVis returns a RenderedHTML wrapper; str() gives the inner HTML."""
    return str(widget)


def build_viewer_html(transcripts: list[dict[str, Any]]) -> str:
    parts: list[str] = [
        "<!doctype html>",
        "<html><head><meta charset='utf-8'>",
        "<title>Per-token probe activations -- Blog 4 supplement</title>",
        "<style>",
        "body { font-family: -apple-system, BlinkMacSystemFont, sans-serif; max-width: 1100px; margin: 2em auto; padding: 0 1em; }",
        "h1, h2 { font-weight: 600; }",
        "details { margin-bottom: 2em; border: 1px solid #ddd; border-radius: 6px; padding: 0.6em 1em; }",
        "summary { cursor: pointer; font-weight: 600; }",
        ".meta { color: #555; font-size: 0.9em; margin-bottom: 0.8em; }",
        "</style></head><body>",
        "<h1>Per-token probe activations -- Blog 4 supplement</h1>",
        "<p>Three Apollo published probes (Roleplaying, Followup, RePE/Honesty) applied to",
        "manually-confirmed AF transcripts. Each token's colour intensity reflects the",
        "probe's score at that position; hover for the exact value.</p>",
    ]

    for r in transcripts:
        summary = (
            f"sample {r['sample_id']} -- {r['manual_label']} -- "
            f"\"{(r.get('advbench_query') or '')[:80]}\""
        )
        parts.append(f"<details><summary>{summary}</summary>")
        parts.append(
            f"<div class='meta'>free-tier: {r.get('is_free_tier_user')}, "
            f"n_tokens: {len(r['tokens'])}, "
            f"sections: {sorted(r.get('sections', {}).keys())}</div>"
        )
        for section in ("scratchpad", "response"):
            widget = render_section(r, section)
            if widget is None:
                continue
            parts.append(f"<h3>{section}</h3>")
            parts.append(_widget_to_html(widget))
        parts.append("</details>")

    parts.append("</body></html>")
    return "\n".join(parts)


html = build_viewer_html(rows)
HTML_OUTPUT_PATH.write_text(html, encoding="utf-8")
print(f"Wrote {HTML_OUTPUT_PATH} ({len(html) / 1024:.1f} KB)")

## 7. Sanity checks

Quick diagnostics to catch the obvious failure modes before publishing:
- Are score distributions sensible (mean ~0, signed both ways)?
- Do the top-scoring tokens match what `inspect_top_tokens.py` printed in
  the cross-probe writeup?
- Are sections aligned to recognisable boundary tokens (`<SCRATCHPAD_REASONING>`)?

In [ ]:
for probe in PROBES:
    all_scores = np.concatenate(
        [np.array(r["scores"][probe], dtype=np.float32) for r in rows]
    )
    print(
        f"{probe:<18}  mean={all_scores.mean():+.4f}  std={all_scores.std():.4f}  "
        f"min={all_scores.min():+.3f}  max={all_scores.max():+.3f}"
    )

# Section boundary spot check on the first selected transcript
if selected:
    r = selected[0]
    for section, (start, end) in (r.get("sections") or {}).items():
        boundary_tokens = r["tokens"][max(0, start - 2):start + 3]
        print(f"{section:<12} [{start}, {end})  start-context: {boundary_tokens!r}")